# SchemaShift on Colab: many open models

Runs a list of open models on the SchemaShift tasks with a free Colab GPU, using the same
`scripts/run_models.py` as on the laptop. Each model is run twice: without and with the
one-line hint.

1. **Runtime -> Change runtime type -> T4 GPU.**
2. Run the cells in order. Step 2 asks for `schemashift_student_club.zip` (scripts, the four
   changed databases, `tasks.jsonl` and the results so far).
3. Step 3 (Google Drive) is strongly recommended: free Colab stops after a few hours, and Drive
   keeps every finished run. Next time, run the same cells again and the loop continues where
   it stopped.
4. Step 8 prints the comparison table, step 9 downloads `results.zip`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project zip and unpack it

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # choose schemashift_student_club.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("/content/Text_To_Sql")
%cd /content/Text_To_Sql
!ls scripts results data/schemashift/student_club

## 3. Keep results in Google Drive (recommended)

Saves results straight into `My Drive/SchemaShift/results`. Results already in the zip are
copied there first, without overwriting anything.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/SchemaShift/results
!cp -n results/* /content/drive/MyDrive/SchemaShift/results/ 2>/dev/null; true
!rm -rf results && ln -s /content/drive/MyDrive/SchemaShift/results results
!ls results

## 4. Install the Python package

In [ ]:
!pip -q install sqlglot==30.23.0

## 5. Install Ollama and start it

The context window is set to 8,192 tokens. With a smaller window, Ollama cuts the start of a
long prompt (the instructions and the schema) without any warning.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, urllib.request

server = subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192"),
                          stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
!ollama --version

## 6. Choose the models

Put `#` in front of a line to skip that model. The list is in priority order, so the most
important comparisons finish first. Rough time per run on a T4: 1.5-8B about 5-10 minutes,
12-16B about 15-20 minutes, reasoning models much longer because they think before answering.

In [ ]:
MODELS = [
    # 1) Size ladder: same family, only the size changes (7b is already done)
    {"name": "qwen2.5-coder:1.5b"},
    {"name": "qwen2.5-coder:3b"},
    {"name": "qwen2.5-coder:7b"},
    {"name": "qwen2.5-coder:14b"},
    # 2) SQL specialists from Hugging Face (trained to write SQL from a question)
    {"name": "hf.co/mradermacher/OmniSQL-7B-GGUF:Q4_K_M"},
    {"name": "hf.co/mradermacher/Arctic-Text2SQL-R1-7B-GGUF:Q4_K_M", "max_tokens": 4096},
    # {"name": "hf.co/mradermacher/XiYanSQL-QwenCoder-7B-2504-GGUF:Q4_K_M"},
    # 3) Other model families
    {"name": "llama3.1:8b"},
    {"name": "gemma3:12b"},
    {"name": "deepseek-coder-v2:16b"},
    {"name": "phi4:14b"},
    # 4) Reasoning models: slow, and they need room for their thinking
    {"name": "qwen3:8b", "max_tokens": 4096},
    {"name": "deepseek-r1:8b", "max_tokens": 4096},
]

## 7. Run every model (safe to stop and run again)

For each model: skip it if both runs are already finished, otherwise download it, run without
and with the hint, then delete it to free disk space. Every finished task is saved at once,
so a stopped session loses nothing.

The cell first checks that steps 2-6 have been run. If Colab restarted the session, it says
which step to run again, and it restarts the Ollama server by itself.

In [ ]:
import datetime, json, os, shutil, subprocess, sys, time, urllib.request

PROJECT = "/content/Text_To_Sql"

# Checks first: each message says which step to run again.
if not os.path.exists(f"{PROJECT}/scripts/run_models.py"):
    raise RuntimeError("Project files not found. Run step 2 (upload the zip) first.")
os.chdir(PROJECT)
if f"{PROJECT}/scripts" not in sys.path:
    sys.path.insert(0, f"{PROJECT}/scripts")
if "MODELS" not in globals():
    raise RuntimeError("MODELS is not defined. Run step 6 first.")
try:
    import sqlglot  # noqa: F401  (run_models.py needs it)
except ImportError:
    raise RuntimeError("sqlglot is missing. Run step 4 first.")
if shutil.which("ollama") is None:
    raise RuntimeError("Ollama is not installed. Run step 5 first.")


LOG = "/content/ollama.log"


def server_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False


def show_log_tail(lines=15):
    # The last lines of the Ollama log usually say why the server stopped (e.g. out of memory).
    try:
        tail = open(LOG, encoding="utf-8", errors="replace").read().splitlines()[-lines:]
        print("   last lines of the Ollama log:")
        for line in tail:
            print("   | " + line)
    except FileNotFoundError:
        pass


def ensure_server():
    # Start the Ollama server if it is not running (session restarted, or it crashed on a model).
    if server_up():
        return True
    print("Ollama server is not running; starting it ...", flush=True)
    show_log_tail()
    subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192"),
                     stdout=open(LOG, "a"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if server_up():
            return True
        time.sleep(1)
    return False


if not ensure_server():
    raise RuntimeError(f"Ollama server did not start. See {LOG}")
print("Checks passed: project files, MODELS, sqlglot, Ollama server.")

from run_models import run_name

TASK_DIR = "data/schemashift/student_club"
N_TASKS = sum(1 for _ in open(f"{TASK_DIR}/tasks.jsonl", encoding="utf-8"))
OLLAMA_VERSION = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()


def finished(model, hint):
    path = f"results/{run_name('student_club', model, 'adapt', hint)}.jsonl"
    try:
        return sum(1 for _ in open(path, encoding="utf-8")) >= N_TASKS
    except FileNotFoundError:
        return False


def stream(cmd):
    # Show the command's output line by line as it arrives (unbuffered, so it does not look stuck).
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


for m in MODELS:
    name, max_tokens = m["name"], m.get("max_tokens", 1024)
    if finished(name, False) and finished(name, True):
        print(f"== {name}: already done, skipping")
        continue
    print()
    print(f"========== {name} ==========")
    if not ensure_server():
        print(f"!! Ollama server will not start; stopping here. See {LOG}")
        break
    print("downloading (big models take 1-3 minutes) ...", flush=True)
    start = time.time()
    pull = subprocess.run(["ollama", "pull", name], capture_output=True, text=True)
    if pull.returncode != 0:
        print(f"!! could not download {name}: {pull.stderr.strip()[-300:]}")
        continue
    print(f"downloaded in {time.time() - start:.0f} s", flush=True)
    for hint in (False, True):
        if finished(name, hint):
            continue
        if not ensure_server():
            print(f"!! Ollama server will not start. See {LOG}")
            break
        print(f"--- {name}, {'with' if hint else 'without'} hint ---", flush=True)
        cmd = [sys.executable, "-u", "scripts/run_models.py", TASK_DIR, "--model", name,
               "--max-tokens", str(max_tokens)] + (["--hint"] if hint else [])
        code = stream(cmd)
        if code != 0:
            print(f"!! the run stopped early (exit code {code}); it will continue next time")
            show_log_tail()
        with open("results/run_log.jsonl", "a", encoding="utf-8") as log:
            print(json.dumps({
                "model": name, "hint": hint, "exit_code": code, "finished": finished(name, hint),
                "ollama": OLLAMA_VERSION,
                "at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            }), file=log)
    subprocess.run(["ollama", "rm", name], capture_output=True)  # free disk for the next model
print()
print("All listed models are done (or skipped).")

## 8. Compare all runs

In [ ]:
!python scripts/compare_results.py results

## 9. Download the results

In [ ]:
!zip -qr /content/results.zip results/
files.download("/content/results.zip")